# 基于多源数据融合的网络攻击检测系统

## 项目指导文档

---

### 目录
1. [项目概述](#1-项目概述)
2. [环境配置](#2-环境配置)
3. [项目结构](#3-项目结构)
4. [数据准备](#4-数据准备)
5. [数据预处理](#5-数据预处理)
6. [模型架构](#6-模型架构)
7. [模型训练](#7-模型训练)
8. [模型评估](#8-模型评估)
9. [实验记录](#9-实验记录)
10. [常见问题](#10-常见问题)

---
## 1. 项目概述

### 1.1 研究背景

网络攻击检测是保护网络安全的重要技术。本项目聚焦于**基于多源数据融合的攻击检测方法**，通过融合网络流量和系统日志等多源数据，利用深度学习技术实现更准确的网络攻击检测。

### 1.2 技术路线

```
多源数据 → 数据预处理 → 特征提取 → 注意力融合 → 攻击分类
   │                                    │
   ├── 网络流量数据 ──────────────────────┤
   └── 系统日志数据 ──────────────────────┘
```

### 1.3 主要特性

- **多源数据融合**: 支持流量+日志等多种数据源
- **注意力机制**: 自动学习不同数据源的重要性权重
- **模块化设计**: 易于扩展和修改
- **Docker支持**: 一键部署，环境隔离

---
## 2. 环境配置

### 2.1 方式一：使用 Docker（推荐）

```bash
# 构建镜像
docker-compose build

# 启动开发环境
docker-compose run attack-detection

# 启动 Jupyter Lab（访问 http://localhost:8889）
docker-compose up jupyter

# 启动 TensorBoard（访问 http://localhost:6007）
docker-compose up tensorboard
```

**启用 GPU 支持**：编辑 `docker-compose.yml`，取消 `deploy` 部分的注释。

### 2.2 方式二：本地环境

```bash
# 创建虚拟环境
python -m venv venv
source venv/bin/activate  # Linux/Mac
# 或 venv\Scripts\activate  # Windows

# 安装依赖
pip install -r requirements.txt
```

---
## 3. 项目结构

```
huanjing/
│
├── data/                       # 数据目录
│   ├── raw/                    # 原始数据集 ⬅️ 放置您的数据
│   ├── processed/              # 预处理后的数据
│   └── logs/                   # 日志类数据
│
├── src/                        # 源代码
│   ├── config/
│   │   └── config.yaml         # 配置文件
│   ├── data/
│   │   └── dataloader.py       # 数据加载模块
│   ├── models/
│   │   └── fusion_net.py       # 深度学习模型
│   ├── utils/
│   │   └── helpers.py          # 工具函数
│   └── train.py                # 训练脚本
│
├── notebooks/                  # Jupyter notebooks
│   └── guide.ipynb             # 本指导文档
│
├── outputs/                    # 输出目录
│   ├── checkpoints/            # 模型权重
│   ├── results/                # 实验结果
│   └── figures/                # 可视化图表
│
├── experiments/                # 实验配置
├── Dockerfile                  # Docker 配置
├── docker-compose.yml          # Docker 编排
├── requirements.txt            # Python 依赖
└── .gitignore                  # Git 忽略规则
```

---
## 4. 数据准备

### 4.1 支持的数据集

本项目支持以下公开数据集（也可使用自定义数据）：

| 数据集 | 描述 | 下载链接 |
|--------|------|----------|
| CICIDS2017 | 包含多种攻击类型的网络流量 | [链接](https://www.unb.ca/cic/datasets/ids-2017.html) |
| NSL-KDD | 经典入侵检测数据集 | [链接](https://www.unb.ca/cic/datasets/nsl.html) |
| UNSW-NB15 | 现代网络流量数据集 | [链接](https://research.unsw.edu.au/projects/unsw-nb15-dataset) |
| CIC-DDoS2019 | DDoS攻击数据集 | [链接](https://www.unb.ca/cic/datasets/ddos-2019.html) |

### 4.2 数据存放位置

```
data/
├── raw/
│   ├── traffic/           # 网络流量数据
│   │   ├── train.csv
│   │   └── test.csv
│   └── logs/              # 系统日志数据
│       ├── train.csv
│       └── test.csv
```

### 4.3 数据格式要求

**流量数据 (CSV)**:
- 每行代表一个网络流/连接
- 最后一列为标签 (label)
- 支持数值型特征

**日志数据 (CSV)**:
- 每行代表一条日志记录
- 需要与流量数据对齐（相同样本数）
- 最后一列为标签 (label)

---
## 5. 数据预处理

以下代码演示如何加载和预处理数据：

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# 配置路径
TRAFFIC_DATA_PATH = '../data/raw/traffic/train.csv'  # 修改为您的数据路径
LOG_DATA_PATH = '../data/raw/logs/train.csv'          # 修改为您的数据路径

In [ ]:
def load_and_preprocess_data(traffic_path, log_path=None):
    """
    加载并预处理数据
    
    参数:
        traffic_path: 流量数据路径
        log_path: 日志数据路径（可选）
    
    返回:
        处理后的特征和标签
    """
    # 加载流量数据
    print("加载流量数据...")
    traffic_df = pd.read_csv(traffic_path)
    print(f"流量数据形状: {traffic_df.shape}")
    
    # 分离特征和标签
    X_traffic = traffic_df.iloc[:, :-1]
    y = traffic_df.iloc[:, -1]
    
    # 处理缺失值
    X_traffic = X_traffic.fillna(0)
    
    # 处理无穷值
    X_traffic = X_traffic.replace([np.inf, -np.inf], 0)
    
    # 标准化
    scaler = StandardScaler()
    X_traffic_scaled = scaler.fit_transform(X_traffic)
    
    # 编码标签
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"类别数: {len(label_encoder.classes_)}")
    print(f"类别: {label_encoder.classes_}")
    
    # 如果有日志数据
    X_log_scaled = None
    if log_path:
        print("\n加载日志数据...")
        log_df = pd.read_csv(log_path)
        X_log = log_df.iloc[:, :-1].fillna(0).replace([np.inf, -np.inf], 0)
        X_log_scaled = scaler.fit_transform(X_log)
        print(f"日志数据形状: {log_df.shape}")
    
    return X_traffic_scaled, X_log_scaled, y_encoded, label_encoder

# 使用示例（取消注释运行）
# X_traffic, X_log, y, label_encoder = load_and_preprocess_data(TRAFFIC_DATA_PATH, LOG_DATA_PATH)

In [ ]:
def split_dataset(X_traffic, X_log, y, test_size=0.2, val_size=0.1):
    """
    划分训练集、验证集、测试集
    """
    # 第一次划分：训练+验证 / 测试
    X_tr_traffic, X_test_traffic, y_train, y_test = train_test_split(
        X_traffic, y, test_size=test_size, random_state=42, stratify=y
    )
    
    # 第二次划分：训练 / 验证
    X_train_traffic, X_val_traffic, y_train, y_val = train_test_split(
        X_tr_traffic, y_train, test_size=val_size, random_state=42, stratify=y_train
    )
    
    # 如果有日志数据，同样划分
    if X_log is not None:
        X_tr_log, X_test_log, _, _ = train_test_split(
            X_log, y, test_size=test_size, random_state=42, stratify=y
        )
        X_train_log, X_val_log, _, _ = train_test_split(
            X_tr_log, y_train, test_size=val_size, random_state=42
        )
    else:
        X_train_log, X_val_log, X_test_log = None, None, None
    
    print(f"训练集大小: {len(y_train)}")
    print(f"验证集大小: {len(y_val)}")
    print(f"测试集大小: {len(y_test)}")
    
    return {
        'train': (X_train_traffic, X_train_log, y_train),
        'val': (X_val_traffic, X_val_log, y_val),
        'test': (X_test_traffic, X_test_log, y_test)
    }

---
## 6. 模型架构

### 6.1 融合网络架构

```
┌─────────────────┐     ┌─────────────────┐
│   流量特征输入    │     │   日志特征输入    │
└────────┬────────┘     └────────┬────────┘
         │                       │
         ▼                       ▼
┌─────────────────┐     ┌─────────────────┐
│   流量编码器      │     │   日志编码器      │
│  (MLP + BN)      │     │  (MLP + BN)      │
└────────┬────────┘     └────────┬────────┘
         │                       │
         └───────────┬───────────┘
                     ▼
           ┌─────────────────┐
           │   注意力融合层    │
           └────────┬────────┘
                    ▼
           ┌─────────────────┐
           │     分类器       │
           └────────┬────────┘
                    ▼
              攻击类型预测
```

### 6.2 模型代码

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentionFusion(nn.Module):
    """注意力融合模块"""
    
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        # x: [batch, num_sources, hidden_dim]
        attn_weights = F.softmax(self.attention(x), dim=1)
        # 加权求和
        return (x * attn_weights).sum(dim=1)


class FusionNet(nn.Module):
    """多源数据融合网络"""
    
    def __init__(self, traffic_dim, log_dim, hidden_dim=256, num_classes=2, dropout=0.3):
        super().__init__()
        
        # 流量特征编码器
        self.traffic_encoder = nn.Sequential(
            nn.Linear(traffic_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        
        # 日志特征编码器
        self.log_encoder = nn.Sequential(
            nn.Linear(log_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        
        # 注意力融合
        self.fusion = AttentionFusion(hidden_dim, hidden_dim // 2)
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    
    def forward(self, traffic_features, log_features):
        # 编码
        traffic_encoded = self.traffic_encoder(traffic_features)
        log_encoded = self.log_encoder(log_features)
        
        # 堆叠并融合
        combined = torch.stack([traffic_encoded, log_encoded], dim=1)
        fused = self.fusion(combined)
        
        # 分类
        return self.classifier(fused)


# 查看模型结构
model = FusionNet(traffic_dim=78, log_dim=50, hidden_dim=256, num_classes=5)
print(model)

---
## 7. 模型训练

### 7.1 训练配置

编辑 `src/config/config.yaml` 修改训练参数：

```yaml
training:
  batch_size: 64        # 批次大小
  epochs: 100           # 训练轮数
  learning_rate: 0.001  # 学习率
  weight_decay: 0.0001  # 权重衰减
  early_stopping_patience: 10  # 早停耐心值
```

### 7.2 训练代码

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

def create_dataloader(X_traffic, X_log, y, batch_size=64, shuffle=True):
    """创建 DataLoader"""
    traffic_tensor = torch.FloatTensor(X_traffic)
    log_tensor = torch.FloatTensor(X_log) if X_log is not None else torch.zeros_like(traffic_tensor)
    y_tensor = torch.LongTensor(y)
    
    dataset = TensorDataset(traffic_tensor, log_tensor, y_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def train_model(model, train_loader, val_loader, config):
    """
    训练模型
    
    参数:
        model: 神经网络模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        config: 训练配置
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"使用设备: {device}")
    
    model = model.to(device)
    
    # 损失函数和优化器
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config['epochs']
    )
    
    # TensorBoard
    writer = SummaryWriter('../outputs/results')
    
    # 早停
    best_val_loss = float('inf')
    patience_counter = 0
    
    # 训练循环
    for epoch in range(config['epochs']):
        # 训练阶段
        model.train()
        train_loss = 0.0
        
        for traffic, log, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            traffic, log, labels = traffic.to(device), log.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(traffic, log)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for traffic, log, labels in val_loader:
                traffic, log, labels = traffic.to(device), log.to(device), labels.to(device)
                outputs = model(traffic, log)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = 100. * correct / total
        
        # 记录到 TensorBoard
        writer.add_scalar('Loss/train', train_loss, epoch)
        writer.add_scalar('Loss/val', val_loss, epoch)
        writer.add_scalar('Accuracy/val', val_acc, epoch)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")
        
        # 早停检查
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # 保存最佳模型
            torch.save(model.state_dict(), '../outputs/checkpoints/best_model.pt')
        else:
            patience_counter += 1
            if patience_counter >= config['early_stopping_patience']:
                print(f"早停: 验证损失连续 {patience_counter} 轮未改善")
                break
        
        scheduler.step()
    
    writer.close()
    return model

In [ ]:
# 训练示例（取消注释运行）

# 配置
# config = {
#     'learning_rate': 0.001,
#     'weight_decay': 0.0001,
#     'epochs': 100,
#     'early_stopping_patience': 10,
#     'batch_size': 64
# }

# 创建数据加载器
# train_loader = create_dataloader(X_train_traffic, X_train_log, y_train, config['batch_size'])
# val_loader = create_dataloader(X_val_traffic, X_val_log, y_val, config['batch_size'], shuffle=False)

# 创建模型
# model = FusionNet(
#     traffic_dim=X_train_traffic.shape[1],
#     log_dim=X_train_log.shape[1],
#     num_classes=len(np.unique(y_train))
# )

# 训练
# trained_model = train_model(model, train_loader, val_loader, config)

### 7.3 命令行训练

```bash
# 使用默认配置训练
python src/train.py

# 使用自定义配置
python src/train.py --config src/config/config.yaml
```

---
## 8. 模型评估

### 8.1 评估指标

- **Accuracy**: 整体准确率
- **Precision**: 精确率（预测为正的样本中实际为正的比例）
- **Recall**: 召回率（实际为正的样本中被正确预测的比例）
- **F1-Score**: 精确率和召回率的调和平均
- **AUC-ROC**: ROC曲线下面积

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc
)
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_model(model, test_loader, label_encoder, device='cpu'):
    """
    评估模型性能
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for traffic, log, labels in test_loader:
            traffic, log = traffic.to(device), log.to(device)
            outputs = model(traffic, log)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # 计算指标
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    print("=" * 50)
    print("模型评估结果")
    print("=" * 50)
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("=" * 50)
    
    # 分类报告
    print("\n详细分类报告:")
    print(classification_report(
        all_labels, all_preds, 
        target_names=label_encoder.classes_
    ))
    
    return all_labels, all_preds, all_probs


def plot_confusion_matrix(y_true, y_pred, classes, save_path=None):
    """
    绘制混淆矩阵
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=classes, yticklabels=classes
    )
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.title('混淆矩阵')
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


# 使用示例（取消注释运行）
# y_true, y_pred, y_probs = evaluate_model(model, test_loader, label_encoder)
# plot_confusion_matrix(y_true, y_pred, label_encoder.classes_, '../outputs/figures/confusion_matrix.png')

---
## 9. 实验记录

### 9.1 使用 TensorBoard

```bash
# 启动 TensorBoard
tensorboard --logdir outputs/results --port 6006

# 或使用 Docker
docker-compose up tensorboard
```

访问 http://localhost:6006 查看训练曲线。

### 9.2 实验对比建议

| 实验 | 描述 | 目的 |
|------|------|------|
| Baseline | 仅使用流量数据的单源模型 | 验证融合的有效性 |
| Concat | 简单拼接融合 | 对比注意力融合 |
| Attention | 注意力融合（本项目） | 主要方法 |
| Ablation | 去除某一数据源 | 分析各源贡献 |

---
## 10. 常见问题

### Q1: CUDA 内存不足

```python
# 减小批次大小
config['batch_size'] = 32  # 或更小

# 使用梯度累积
accumulation_steps = 4
```

### Q2: 数据不平衡

```python
from sklearn.utils.class_weight import compute_class_weight

# 计算类别权重
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
weights = torch.FloatTensor(class_weights).to(device)

# 使用加权损失
criterion = nn.CrossEntropyLoss(weight=weights)
```

### Q3: 如何只使用单一数据源？

```python
from src.models.fusion_net import SingleSourceNet

# 使用单源网络
model = SingleSourceNet(
    input_dim=X_traffic.shape[1],
    num_classes=num_classes
)
```

### Q4: 如何添加新的数据源？

修改 `FusionNet` 类，添加新的编码器，并在 `forward` 方法中将新编码结果加入融合。

### Q5: Docker 构建失败

```bash
# 清理缓存重新构建
docker-compose build --no-cache

# 检查 Docker 版本
docker --version
docker-compose --version
```

---

## 快速开始清单

- [ ] 1. 配置环境（Docker 或本地）
- [ ] 2. 将数据集放入 `data/raw/` 目录
- [ ] 3. 修改配置文件 `src/config/config.yaml`
- [ ] 4. 运行数据预处理
- [ ] 5. 训练模型
- [ ] 6. 评估模型
- [ ] 7. 查看 TensorBoard 结果

---

如有问题，请检查：
1. 数据格式是否正确
2. 配置文件路径是否正确
3. GPU 驱动是否安装（如使用GPU）